# CreditWise AI

## Home Credit Default Risk Prediction

### Objective

Develop an AI-powered credit risk assessment system capable of predicting loan default probability using customer financial and behavioral attributes.

### Dataset

Home Credit Default Risk Dataset

### Problem Type

Binary Classification

### Target Variable

- 0 = Loan Repaid
- 1 = Loan Default

## 1. Dataset Loading and Initial Verification

Load the Home Credit training dataset and verify the number of observations and available features.

In [ ]:
import pandas as pd

train_df = pd.read_csv("../data/raw/application_train.csv")

print(train_df.shape)

## 2. Dataset Preview

Inspect a sample of records to understand feature structure and data representation.

In [ ]:
train_df.head()

## 3. Dataset Information

Analyze column data types, non-null counts, and overall dataset structure.

In [ ]:
train_df.info()

## 4. Target Variable Analysis

Examine the distribution of loan repayment and loan default records.

In [ ]:
train_df["TARGET"].value_counts()

## 5. Target Distribution Percentage Analysis

Calculate the percentage of defaulting and non-defaulting customers to assess class imbalance.

In [ ]:
train_df["TARGET"].value_counts(normalize=True) * 100

## 6. Feature Inventory

Review all available dataset features and understand the breadth of information provided.

In [ ]:
train_df.columns.tolist()

## 7. Feature Type Analysis

Determine the number of numerical and categorical features available for modeling.

In [ ]:
train_df.dtypes.value_counts()

## 8. Missing Value Assessment (Absolute Count)

Identify columns containing missing values and quantify the number of missing observations.

In [ ]:
missing = train_df.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)

missing.head(20)

## 9. Missing Value Assessment (Percentage)

Evaluate the severity of missing data by calculating the percentage of missing values per feature.

In [ ]:
missing_percent = (train_df.isnull().sum() / len(train_df)) * 100

missing_percent = missing_percent[missing_percent > 0]

missing_percent.sort_values(ascending=False).head(20)

## 10. Target Distribution Visualization

Visualize the imbalance between loan repayment and loan default classes.

In [ ]:
import matplotlib.pyplot as plt

target_counts = train_df["TARGET"].value_counts()

plt.figure(figsize=(6,4))
target_counts.plot(kind="bar")

plt.title("Loan Repayment vs Default Distribution")
plt.xlabel("Target")
plt.ylabel("Count")

plt.show()

## Key Findings

### Dataset Characteristics

- Total Records: 307,511
- Total Features: 122
- Numerical Features: 106
- Categorical Features: 16

### Target Distribution

- Loan Repaid (0): 91.93%
- Loan Default (1): 8.07%

### Initial Observations

- The dataset exhibits significant class imbalance.
- Accuracy alone will not be an appropriate evaluation metric.
- Several housing-related features contain more than 60% missing values.
- Feature selection and missing value treatment will be critical preprocessing steps.
- Metrics such as ROC-AUC, Precision, Recall, and F1-score will be considered during model evaluation.

# Feature Quality Assessment

The objective of this phase is to evaluate feature usefulness, identify low-quality columns, and develop a data preprocessing strategy for model training.

## High Missing Value Feature Identification

Features with extremely high missingness can negatively affect model quality and increase preprocessing complexity.

This analysis identifies features with more than 60% missing values for further evaluation.

In [ ]:
high_missing = missing_percent[missing_percent > 60]

print(f"Number of features with >60% missing values: {len(high_missing)}")

high_missing.sort_values(ascending=False)

## Feature Retention Assessment

Estimate the number of features that would remain if highly incomplete columns were removed.

This helps evaluate whether aggressive feature elimination would significantly reduce the information available for modeling.

In [ ]:
total_features = train_df.shape[1] - 1   #excluding TARGET

remaining_features = total_features - len(high_missing)

print("Total Features:", total_features)
print("Features >60% Missing:", len(high_missing))
print("Remaining Features:", remaining_features)

## Categorical Feature Identification

Machine learning algorithms require categorical variables to be transformed into numerical representations before model training.

This analysis identifies all categorical features present in the dataset and provides an overview of the variables that will require encoding during preprocessing.

In [ ]:
categorical_cols = train_df.select_dtypes(include=["object", "string"]).columns.tolist()

print("Number of categorical features:", len(categorical_cols))
categorical_cols

## Categorical Feature Cardinality Analysis

Cardinality refers to the number of unique values present within a categorical feature.

Understanding feature cardinality is important because it influences the choice of encoding strategy. Low-cardinality features are often suitable for one-hot encoding, while high-cardinality features may require alternative approaches to avoid excessive dimensionality.

In [ ]:
for col in categorical_cols:
    print(f"{col}: {train_df[col].nunique()}")

## Observation: Categorical Feature Cardinality

Most categorical features in the dataset exhibit low cardinality, with fewer than 20 unique values. Such features can typically be encoded efficiently using one-hot encoding without significantly increasing dimensionality.

One feature, `ORGANIZATION_TYPE`, contains 58 unique categories and represents a relatively high-cardinality variable. Applying one-hot encoding directly to this feature may introduce a large number of additional columns and increase model complexity.

During the preprocessing phase, low-cardinality features will be considered for one-hot encoding, while alternative encoding techniques such as frequency encoding will be evaluated for `ORGANIZATION_TYPE`.


# Statistical Feature Analysis

The objective of this phase is to investigate relationships between individual features and loan default behavior.

Understanding these relationships helps identify predictive signals, guide feature engineering decisions, and improve model interpretability.

## Missingness Signal Analysis: Vehicle Age

Missing values are not always random.

This analysis investigates whether the absence of vehicle age information is associated with different loan default behavior.

If default rates differ significantly, missingness itself may become a useful predictive feature.

In [ ]:
train_df["OWN_CAR_AGE_MISSING"] = train_df["OWN_CAR_AGE"].isnull().astype(int)

train_df.groupby("OWN_CAR_AGE_MISSING")["TARGET"].mean()

## Income Analysis by Default Status

Income level is one of the most important factors in credit risk assessment.

This analysis compares average applicant income between customers who repaid loans and those who defaulted.

In [ ]:
train_df.groupby("TARGET")["AMT_INCOME_TOTAL"].mean()

## Income Distribution Visualization

Visualize the distribution of applicant income across repayment outcomes.

The objective is to identify potential differences, outliers, and income-related risk patterns.

In [ ]:
import matplotlib.pyplot as plt

train_df.boxplot(
    column="AMT_INCOME_TOTAL",
    by="TARGET",
    figsize=(8,5)
)

plt.title("Income Distribution by Default Status")
plt.suptitle("")
plt.show()

## Credit Amount Analysis

Loan size can influence repayment risk.

This analysis compares average credit amounts between defaulting and non-defaulting customers.

In [ ]:
train_df.groupby("TARGET")["AMT_CREDIT"].mean()

## Credit Amount Distribution Visualization

Visualize the distribution of requested credit amounts across repayment outcomes.

This helps identify whether larger or smaller loans are associated with higher default risk.

In [ ]:
train_df.boxplot(
    column="AMT_CREDIT",
    by="TARGET",
    figsize=(8,5)
)

plt.title("Credit Amount by Default Status")
plt.suptitle("")
plt.show()

## Gender-Based Default Analysis

Evaluate whether default behavior differs across applicant gender categories.

This analysis provides an initial understanding of demographic risk patterns within the dataset.

In [ ]:
pd.crosstab(
    train_df["CODE_GENDER"],
    train_df["TARGET"],
    normalize="index"
)

## Education Level and Default Risk

Educational attainment often correlates with income stability and employment opportunities.

This analysis investigates how default rates vary across education categories.

In [ ]:
pd.crosstab(
    train_df["NAME_EDUCATION_TYPE"],
    train_df["TARGET"],
    normalize="index"
).sort_values(1, ascending=False)

## Income Source and Default Risk

Income source may influence repayment reliability.

This analysis compares default rates across different employment and income categories to identify higher-risk applicant groups.

In [ ]:
pd.crosstab(
    train_df["NAME_INCOME_TYPE"],
    train_df["TARGET"],
    normalize="index"
).sort_values(1, ascending=False)

## Statistical Analysis Findings

### Missing Value Signal

Applicants with missing vehicle-age information exhibited a higher default rate than applicants with available vehicle-age data. This suggests that missingness itself may contain predictive information and should be considered during feature engineering.

### Income and Credit Amount

Average income and average credit amount showed only modest differences between repayment and default groups. These features may still contribute predictive value when combined with other variables.

### Gender

Male applicants exhibited a higher default rate than female applicants, indicating potential predictive value.

### Education

Default rates decreased consistently with increasing education level. Applicants with higher educational attainment demonstrated substantially lower default risk.

### Income Type

Default behavior varied across income categories, suggesting that employment and income source are important risk indicators.

### Preliminary Conclusion

Education level, income type, gender, and missing-value indicators appear to be promising predictive features and will be retained for future modeling and feature engineering.